# 10 — Analysis Dashboard

Analyse the Irish web schema.org audit results and generate reports.

**Key questions**:
1. What % of Irish SMEs have schema markup?
2. Of those that do, what's the average quality?
3. What schema types dominate? (LocalBusiness vs Product vs Article etc.)
4. What's the improvement delta from our model vs existing markup?
5. Which industries/sectors are most underserved?

**Outputs**: Charts + summary stats for the national schema.org audit report

In [ ]:
import sys
sys.path.insert(0, '..')

import json
from pathlib import Path
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')

RESULTS_PATH = Path('../data/results/irish_web_results.jsonl')

if not RESULTS_PATH.exists():
    print('No results found. Run notebook 09 first.')
else:
    results = []
    with open(RESULTS_PATH) as f:
        for line in f:
            results.append(json.loads(line))
    print(f'Loaded {len(results):,} domain results')
    df = pd.DataFrame(results)

## 1. Schema Adoption Overview

In [ ]:
if 'results' in dir() and results:
    n_total = len(results)
    n_had_schema = sum(1 for r in results if r.get('existing_schema'))
    n_no_schema = n_total - n_had_schema
    n_generated = sum(1 for r in results if r.get('generated_schema'))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Adoption pie
    axes[0].pie(
        [n_had_schema, n_no_schema],
        labels=[f'Has schema\n({n_had_schema:,})', f'No schema\n({n_no_schema:,})'],
        autopct='%1.1f%%',
        colors=['#2ecc71', '#e74c3c'],
        startangle=90,
    )
    axes[0].set_title(f'Schema.org Adoption\nIrish Web ({n_total:,} domains)')

    # Quality score distribution
    quality_scores = [r['quality_score'] for r in results if r.get('quality_score', 0) > 0]
    axes[1].hist(quality_scores, bins=30, color='#3498db', edgecolor='white', alpha=0.8)
    axes[1].set_xlabel('Quality Score')
    axes[1].set_ylabel('Number of Domains')
    axes[1].set_title('Generated Schema Quality Distribution')
    axes[1].axvline(x=0.4, color='orange', linestyle='--', label='Min threshold')
    axes[1].axvline(x=0.7, color='green', linestyle='--', label='High quality')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig('../data/results/adoption_overview.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f'Total domains processed: {n_total:,}')
    print(f'Had existing schema: {n_had_schema:,} ({100*n_had_schema/n_total:.1f}%)')
    print(f'Had no schema: {n_no_schema:,} ({100*n_no_schema/n_total:.1f}%)')
    print(f'Model generated valid schema: {n_generated:,} ({100*n_generated/n_total:.1f}%)')

## 2. Schema Type Distribution

In [ ]:
if 'results' in dir() and results:
    type_counter = Counter()
    for r in results:
        for t in r.get('schema_types', []):
            type_counter[t] += 1

    top_types = type_counter.most_common(15)
    types, counts = zip(*top_types) if top_types else ([], [])

    fig, ax = plt.subplots(figsize=(12, 6))
    bars = ax.barh(range(len(types)), counts, color='#3498db', alpha=0.85)
    ax.set_yticks(range(len(types)))
    ax.set_yticklabels(types)
    ax.invert_yaxis()
    ax.set_xlabel('Number of Domains')
    ax.set_title('Top Schema.org Types — Irish Web')

    for bar, count in zip(bars, counts):
        ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
                f'{count:,}', va='center', fontsize=9)

    plt.tight_layout()
    plt.savefig('../data/results/schema_types.png', dpi=150, bbox_inches='tight')
    plt.show()

## 3. Improvement Delta Analysis

In [ ]:
if 'results' in dir() and results:
    improved = [
        r for r in results
        if r.get('existing_schema') and r.get('improvement_delta')
    ]

    if improved:
        deltas = [r['improvement_delta'].get('quality_delta', 0) for r in improved]
        new_props = [len(r['improvement_delta'].get('new_properties', [])) for r in improved]

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        axes[0].hist(deltas, bins=30, color='#2ecc71', edgecolor='white', alpha=0.8)
        axes[0].axvline(x=0, color='red', linestyle='--', label='No change')
        axes[0].set_xlabel('Quality Score Delta')
        axes[0].set_ylabel('Count')
        axes[0].set_title('Quality Improvement vs Existing Schema')
        axes[0].legend()

        axes[1].hist(new_props, bins=20, color='#e67e22', edgecolor='white', alpha=0.8)
        axes[1].set_xlabel('New Properties Added')
        axes[1].set_ylabel('Count')
        axes[1].set_title('New Properties Added by Model')

        plt.tight_layout()
        plt.savefig('../data/results/improvement_delta.png', dpi=150, bbox_inches='tight')
        plt.show()

        print(f'Domains with existing schema analysed: {len(improved):,}')
        print(f'Average quality delta: {sum(deltas)/len(deltas):+.3f}')
        print(f'Average new properties added: {sum(new_props)/len(new_props):.1f}')
        improved_count = sum(1 for d in deltas if d > 0.05)
        print(f'Domains meaningfully improved (>0.05 quality delta): {improved_count:,} ({100*improved_count/len(improved):.1f}%)')

## 4. Sample Results — Site-Level Deep Dives

In [ ]:
if 'results' in dir() and results:
    # Show top 10 most improved sites
    improved_sorted = sorted(
        [r for r in results if r.get('improvement_delta') and r['improvement_delta'].get('quality_delta', 0) > 0],
        key=lambda x: x['improvement_delta'].get('quality_delta', 0),
        reverse=True,
    )

    print('Top 10 most improved sites:')
    for r in improved_sorted[:10]:
        delta = r['improvement_delta']
        print(f"  {r['domain']}")
        print(f"    Quality: {r['quality_score']:.3f} (+{delta.get('quality_delta', 0):+.3f})")
        print(f"    New properties: {delta.get('new_properties', [])}")

## 5. Export Summary Report

In [ ]:
if 'results' in dir() and results:
    summary = {
        'total_domains_processed': len(results),
        'domains_with_existing_schema': sum(1 for r in results if r.get('existing_schema')),
        'domains_without_schema': sum(1 for r in results if not r.get('existing_schema')),
        'schema_adoption_rate': round(sum(1 for r in results if r.get('existing_schema')) / len(results), 3),
        'valid_schema_generated': sum(1 for r in results if r.get('generated_schema')),
        'avg_quality_score': round(sum(r.get('quality_score', 0) for r in results if r.get('generated_schema')) / max(sum(1 for r in results if r.get('generated_schema')), 1), 3),
        'avg_properties_count': round(sum(r.get('properties_count', 0) for r in results if r.get('generated_schema')) / max(sum(1 for r in results if r.get('generated_schema')), 1), 1),
        'top_schema_types': dict(Counter(t for r in results for t in r.get('schema_types', [])).most_common(10)),
    }

    summary_path = Path('../data/results/irish_web_summary.json')
    with open(summary_path, 'w') as f:
        json.dump(summary, f, indent=2)

    print('Summary report:')
    for k, v in summary.items():
        if k != 'top_schema_types':
            print(f'  {k}: {v}')
    print(f'\nSaved to {summary_path}')
    print('\nAnalysis complete. See data/results/ for all outputs.')